In [0]:
%sql create schema if not exists apexlife.silver ;

In [0]:
%sql select * from apexlife.bronze.visits_raw ;

visit_id,patient_id,hospital_id,admission_date,discharge_date,diagnosis_code,cost,_rescued_data
V1001,P001,H003,2025-01-01,2025-01-05,D003,12000,null
V1002,P001,H003,2025-01-20,2025-01-22,D003,9000,null
V1003,P002,H002,2025-02-10,2025-02-15,D001,18000,null
V1004,P002,H002,2025-04-01,2025-04-05,D001,17000,null
V1005,P003,H004,2025-03-12,2025-03-18,D005,22000,null
V1006,P003,H004,2025-03-28,2025-03-31,D005,16000,null
V1007,P004,H005,2025-01-15,2025-01-20,D004,14000,null
V1008,P005,H001,2025-02-25,2025-03-02,D002,20000,null


In [0]:
bronze_table = 'apexlife.bronze.visits_raw'
silver_table = 'apexlife.silver.fact_visit'

checkpoint_path = "abfss://data@apexlife.dfs.core.windows.net/silver/fact_visit/checkpoint/"

In [0]:
from pyspark.sql.functions import  * 

In [0]:
df = (
    spark.readStream.table(bronze_table)
)

In [0]:
df = (
    df
    .dropDuplicates(['visit_id'])
    .withColumn('load_timestamp', current_timestamp())
)

In [0]:
df_hospital  = spark.read.table('apexlife.silver.dim_hospital')
df_patient   = spark.read.table('apexlife.silver.dim_patient' )
df_diagnosis = spark.read.table('apexlife.silver.dim_diagnosis')

In [0]:
# Rename columns to avoid duplicates
df_patient  = df_patient.withColumnRenamed("city" ,  "patient_city")
df_hospital = df_hospital.withColumnRenamed("city", "hospital_city")

In [0]:
df_fact = (
    df
    .join(df_patient ,  on = 'patient_id' , how = 'left')
    .join(df_hospital,  on = 'hospital_id', how = 'left')
    .join(df_diagnosis, on = 'diagnosis_code', how = 'left')
    .withColumn("admission_date", to_date("admission_date"))
    .withColumn("discharge_date", to_date("discharge_date"))
)

In [0]:
from delta.tables import DeltaTable

In [0]:
# Clear stale checkpoint so the stream reprocesses all available source data
dbutils.fs.rm(checkpoint_path, recurse=True)

def merge_fact_visit(batch_df, batch_id):

    if not spark.catalog.tableExists(silver_table):
        batch_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
        return

    fact = DeltaTable.forName(spark, silver_table)
    fact.alias("t").merge(
        batch_df.alias("s"),
        "t.visit_id = s.visit_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()


(
    df_fact
    .drop("load_timestamp")
    .drop("_rescued_data")
    .drop("load_time")
    .writeStream
    .foreachBatch(merge_fact_visit)
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .start()
)

In [0]:

%sql select * from apexlife.silver.fact_visit ;

diagnosis_code,hospital_id,patient_id,visit_id,admission_date,discharge_date,cost,first_name,last_name,gender,dob,patient_city,patient_first_last_name_masked,hospital_name,hospital_city,bed_count,diagnosis_desc
D003,H003,P001,V1001,2025-01-01,2025-01-05,12000,Aarav,Sharma,M,1980-02-14,Mumbai,c0506dd98205aaa3324e64f0a1745d8f525fb38e9d4a310d64444dd9c6b7798c,Kokilaben Dhirubhai Hospital,Mumbai,700,Chest Pain
D005,H004,P003,V1005,2025-03-12,2025-03-18,22000,Kabir,Menon,M,1990-11-02,Bangalore,5216e63785594d7f8efd937c6360ec4317a3285f84632b2f7b754b67b650ea7e,Manipal Hospital,Bangalore,400,Kidney Infection
D005,H004,P003,V1006,2025-03-28,2025-03-31,16000,Kabir,Menon,M,1990-11-02,Bangalore,5216e63785594d7f8efd937c6360ec4317a3285f84632b2f7b754b67b650ea7e,Manipal Hospital,Bangalore,400,Kidney Infection
D003,H003,P001,V1002,2025-01-20,2025-01-22,9000,Aarav,Sharma,M,1980-02-14,Mumbai,c0506dd98205aaa3324e64f0a1745d8f525fb38e9d4a310d64444dd9c6b7798c,Kokilaben Dhirubhai Hospital,Mumbai,700,Chest Pain
D002,H001,P005,V1008,2025-02-25,2025-03-02,20000,Vikram,Singh,M,1965-05-05,Chennai,deb97260b70ff54088f4b6cf6f75edf041b65eed5b26c608290dabc0e9efe0b6,Apollo Main Hospital,Chennai,850,Diabetes Type 2
D001,H002,P002,V1004,2025-04-01,2025-04-05,17000,Riya,Verma,F,1975-07-22,Delhi,80d7212c916fd48a06f7ca020b57ca48b2d8dc1d70151a2a1ad28205e40e5a02,Fortis Healthcare Delhi,Delhi,600,Hypertension
D001,H002,P002,V1003,2025-02-10,2025-02-15,18000,Riya,Verma,F,1975-07-22,Delhi,80d7212c916fd48a06f7ca020b57ca48b2d8dc1d70151a2a1ad28205e40e5a02,Fortis Healthcare Delhi,Delhi,600,Hypertension
D004,H005,P004,V1007,2025-01-15,2025-01-20,14000,Sneha,Rao,F,1988-09-19,Hyderabad,78a3e0a772632903fef2ffe51d6119d7df94f79dfb3295be92ecafe5029dc20f,Yashoda Hospital,Hyderabad,450,Asthma
